In [1]:
%pip install azure-ai-textanalytics azure-core python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv
from azure.ai.textanalytics import TextAnalyticsClient
from azure.core.credentials import AzureKeyCredential

# Load environment variables from .env file
load_dotenv()

endpoint = os.getenv("AZURE_LANGUAGE_ENDPOINT")
key = os.getenv("AZURE_LANGUAGE_KEY")

# Initialize the client
client = TextAnalyticsClient(
    endpoint=endpoint, 
    credential=AzureKeyCredential(key)
)

# Test with a simple sentiment analysis call
documents = [
    "The service was excellent.",
    "The delivery was very slow.",
    "The product is acceptable."
]

results = client.analyze_sentiment(documents=documents)

for text, result in zip(documents, results):
    if not result.is_error:
        print(text, "->", result.sentiment)

The service was excellent. -> positive
The delivery was very slow. -> negative
The product is acceptable. -> positive


In [3]:
import os
from dotenv import load_dotenv

from azure.ai.vision.imageanalysis import ImageAnalysisClient
from azure.ai.vision.imageanalysis.models import VisualFeatures
from azure.core.credentials import AzureKeyCredential
from azure.core.exceptions import HttpResponseError

# 1. Read Azure credentials from environment variables
load_dotenv()
endpoint = os.environ["VISION_ENDPOINT"]
key = os.environ["VISION_KEY"]

# 2. Connect to the Azure Computer Vision resource
client = ImageAnalysisClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(key)
)

# 3. Specify the local image path
image_path = "sample.jpg"

try:
    # 4. Read image bytes
    with open(image_path, "rb") as image_file:
        image_data = image_file.read()

    # 5. Send the image to Azure for Analysis
    result = client.analyze(
        image_data=image_data,
        visual_features=[
            VisualFeatures.CAPTION,
            VisualFeatures.READ
        ],
        gender_neutral_caption=True
    )

    # 6. Output Caption Results
    print("\n--- Image Recognition Results ---")

    if result.caption:
        print("\nImage Description:")
        print(result.caption.text)
        print(f"Confidence: {result.caption.confidence:.2%}")

    # 7. Extract text from the image using OCR
    if result.read:
        print("\nText Detected in Image:")

        for block in result.read.blocks:
            for line in block.lines:
                print(line.text)

    print("\nImage analysis completed successfully.")

except FileNotFoundError:
    print(f"Image file not found: {image_path}")
except HttpResponseError as e:
    print(f"Azure API call failed: {e}")


--- Image Recognition Results ---

Image Description:
a couple of kids playing football
Confidence: 73.11%

Text Detected in Image:
dobit Stock
Adobe Stock | #211758680

Image analysis completed successfully.


In [4]:
# Azure Immersive Reader

In [5]:
%pip install flask requests python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import requests
from flask import Flask, jsonify, render_template_string, request
from dotenv import load_dotenv

# Load environment variables (override=True ensures updates to .env are picked up on re-runs)
load_dotenv(override=True)

app = Flask(__name__)

# Azure credentials
TENANT_ID = os.environ["TENANT_ID"]
CLIENT_ID = os.environ["CLIENT_ID"]
CLIENT_SECRET = os.environ["CLIENT_SECRET"]
SUBDOMAIN = os.environ["SUBDOMAIN"]

# HTML frontend
HTML_PAGE = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <title>Azure Immersive Reader</title>
    <script src="https://ircdname.azureedge.net/immersivereadersdk/immersive-reader-sdk.1.4.0.js"></script>
    <style>
        body {
            font-family: Arial;
            margin: 40px;
            background: #f5f5f5;
        }
        .container {
            background: white;
            padding: 30px;
            border-radius: 10px;
            max-width: 800px;
            margin: auto;
        }
        textarea {
            width: 100%;
            height: 180px;
            padding: 10px;
            font-size: 16px;
            box-sizing: border-box;
        }
        button {
            padding: 12px 20px;
            background: #0078d4;
            color: white;
            border: none;
            cursor: pointer;
            margin-top: 15px;
            border-radius: 5px;
        }
        button:hover {
            background: #005a9e;
        }
        #status {
            margin-top: 15px;
        }
    </style>
</head>
<body>
<div class="container">
    <h1>Azure Immersive Reader</h1>
    <p>Enter some text below and launch Microsoft Immersive Reader.</p>
    <textarea id="reading-content">Machine learning enables computers to learn patterns from data and make predictions.

Artificial intelligence is transforming industries by automating tasks and improving decision-making.

Python is widely used for machine learning and artificial intelligence development.</textarea>
    <br>
    <button onclick="launchReader()">Open Immersive Reader</button>
    <p id="status"></p>
</div>

<script>
async function launchReader() {
    const status = document.getElementById("status");
    status.innerText = "Obtaining Azure access token...";
    try {
        const response = await fetch("/token", { cache: "no-store" });
        if (!response.ok) {
            throw new Error("Authentication failed");
        }
        const auth = await response.json();
        const readingText = document.getElementById("reading-content").value;

        const content = {
            title: "My Reading Content",
            chunks: [
                {
                    content: readingText,
                    lang: "en"
                }
            ]
        };

        status.innerText = "Launching Immersive Reader...";
        await ImmersiveReader.launchAsync(auth.token, auth.subdomain, content);
        status.innerText = "Immersive Reader closed.";
    } catch (error) {
        status.innerText = "Error: " + error.message;
        console.error(error);
    }
}
</script>
</body>
</html>
"""

@app.route("/")
def index():
    return render_template_string(HTML_PAGE)

@app.route("/token")
def get_token():
    token_url = f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token"
    payload = {
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
        "scope": "https://cognitiveservices.azure.com/.default",
        "grant_type": "client_credentials"
    }

    try:
        response = requests.post(token_url, data=payload, timeout=20)
        response.raise_for_status()
        token_data = response.json()
        return jsonify({
            "token": token_data["access_token"],
            "subdomain": SUBDOMAIN
        })
    except requests.RequestException as e:
        error_detail = e.response.text if e.response is not None else str(e)
        print("Azure error:", error_detail)
        return jsonify({"error": "Unable to authenticate with Azure", "details": error_detail}), 502

if __name__ == "__main__":
    app.run(host="127.0.0.1", port=5000, debug=False, use_reloader=False)

In [6]:
# Python implementation: Analyze an image

In [7]:
# Install the required libraries
%pip install azure-ai-vision-imageanalysis python-dotenv

import os
from dotenv import load_dotenv
from azure.ai.vision.imageanalysis import ImageAnalysisClient
from azure.ai.vision.imageanalysis.models import VisualFeatures
from azure.core.credentials import AzureKeyCredential

# Load environment variables from .env file
load_dotenv()

# Azure credentials
ENDPOINT = os.getenv("VISION_ENDPOINT")
KEY = os.getenv("VISION_KEY")

# Create Azure Vision client
client = ImageAnalysisClient(
    endpoint=ENDPOINT,
    credential=AzureKeyCredential(KEY)
)

# Image URL
image_url = "https://aka.ms/azsdk/image-analysis/sample.jpg"

# Analyze image
result = client.analyze_from_url(
    image_url=image_url,
    visual_features=[
        VisualFeatures.CAPTION,
        VisualFeatures.TAGS,
        VisualFeatures.OBJECTS
    ]
)

# Print image description
if result.caption:
    print("Image Description:")
    print(result.caption.text)

# Print tags
if result.tags:
    print("\nDetected Tags:")
    for tag in result.tags.list:
        print(tag.name, tag.confidence)

# Print detected objects
if result.objects:
    print("\nDetected Objects:")
    for obj in result.objects.list:
        print("Object:", obj.tags[0].name)
        print("Confidence:", obj.tags[0].confidence)
        print("Bounding Box:", obj.bounding_box)

Note: you may need to restart the kernel to use updated packages.
Image Description:
a woman wearing a mask sitting at a table with a laptop

Detected Tags:
furniture 0.9874444007873535
clothing 0.9792501926422119
person 0.9427268505096436
houseplant 0.9400021433830261
desk 0.9182863235473633
indoor 0.8963587880134583
laptop 0.8781791925430298
computer 0.8481509685516357
sitting 0.8134787678718567
wall 0.7511621117591858
woman 0.7410725951194763
table 0.6811204552650452
plant 0.6445217132568359
using 0.5358914136886597

Detected Objects:
Object: chair
Confidence: 0.618
Bounding Box: {'x': 603, 'y': 225, 'w': 152, 'h': 224}
Object: person
Confidence: 0.881
Bounding Box: {'x': 399, 'y': 244, 'w': 249, 'h': 325}
Object: Laptop
Confidence: 0.767
Bounding Box: {'x': 295, 'y': 387, 'w': 211, 'h': 102}
Object: chair
Confidence: 0.581
Bounding Box: {'x': 441, 'y': 436, 'w': 256, 'h': 136}
Object: dining table
Confidence: 0.606
Bounding Box: {'x': 123, 'y': 437, 'w': 460, 'h': 125}


In [8]:
# Video Indexer

In [25]:

import os
import json
import base64
import requests

from pathlib import Path
from dotenv import load_dotenv
from azure.identity import InteractiveBrowserCredential

# ============================================================
# STEP 0: Load environment variables
# ============================================================

env_path = Path.cwd() / ".env"

if not env_path.is_file():
    raise FileNotFoundError(
        f".env file not found at: {env_path}"
    )

load_dotenv(dotenv_path=env_path, override=True)

SUBSCRIPTION_ID = os.environ["AZURE_SUBSCRIPTION_ID"]
RESOURCE_GROUP = os.environ["AZURE_RESOURCE_GROUP"]
ACCOUNT_NAME = os.environ["VI_ACCOUNT_NAME"]
ACCOUNT_ID = os.environ["VI_ACCOUNT_ID"]
LOCATION = os.environ["VI_LOCATION"]
VIDEO_ID = os.environ["VI_VIDEO_ID"]
TENANT_ID = os.environ["AZURE_TENANT_ID"]

print("Environment file:", env_path)
print("Video Indexer account ID:", ACCOUNT_ID)
print("Region:", LOCATION)
print("Video ID:", VIDEO_ID)


# ============================================================
# STEP 1: Authenticate with Azure Resource Manager
# ============================================================

credential = InteractiveBrowserCredential(
    tenant_id=TENANT_ID
)

arm_token = credential.get_token(
    "https://management.azure.com/.default"
).token

print("\nAzure ARM authentication successful!")


# ============================================================
# STEP 2: Generate Video Indexer access token
# ============================================================

arm_url = (
    f"https://management.azure.com/subscriptions/"
    f"{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}"
    f"/providers/Microsoft.VideoIndexer/accounts/"
    f"{ACCOUNT_NAME}/generateAccessToken"
)

response = requests.post(
    arm_url,
    headers={
        "Authorization": f"Bearer {arm_token}",
        "Content-Type": "application/json"
    },
    params={
        "api-version": "2024-01-01"
    },
    json={
        "permissionType": "Reader",
        "scope": "Account"
    },
    timeout=30
)

print("ARM token request status:", response.status_code)

if not response.ok:
    print("ARM error:", response.text)
    raise RuntimeError(
        f"Video Indexer token request failed: "
        f"HTTP {response.status_code}"
    )

access_token = response.json()["accessToken"]

print("Video Indexer access token generated successfully!")


# ============================================================
# STEP 3: Verify account ID inside the token
# ============================================================

token_payload = access_token.split(".")[1]
token_payload += "=" * (-len(token_payload) % 4)

token_claims = json.loads(
    base64.urlsafe_b64decode(token_payload)
)

TOKEN_ACCOUNT_ID = token_claims.get("AccountId")

print("\nAccount verification:")
print("Account ID from .env:", ACCOUNT_ID)
print("Account ID from token:", TOKEN_ACCOUNT_ID)

if ACCOUNT_ID.lower() != str(TOKEN_ACCOUNT_ID).lower():
    raise ValueError(
        "ACCOUNT ID MISMATCH!\n"
        "Your VI_ACCOUNT_ID does not match the Azure "
        "Video Indexer resource.\n"
        f"Set VI_ACCOUNT_ID={TOKEN_ACCOUNT_ID} "
        "in your .env file and rerun this cell."
    )

print("Account ID verification passed!")


# ============================================================
# STEP 4: Retrieve video indexing information
# ============================================================

BASE_URL = "https://api.videoindexer.ai"

index_url = (
    f"{BASE_URL}/{LOCATION}/Accounts/"
    f"{ACCOUNT_ID}/Videos/{VIDEO_ID}/Index"
)

result = requests.get(
    index_url,
    params={
        "accessToken": access_token,
        "language": "English"
    },
    timeout=60
)

print("\nVideo indexing HTTP status:", result.status_code)

if not result.ok:
    print("Video Indexer error:", result.text)
    raise RuntimeError(
        f"Failed to retrieve video insights: "
        f"HTTP {result.status_code}"
    )

video_data = result.json()

print("Video indexing information retrieved successfully!")


# ============================================================
# STEP 5: Display video transcription
# ============================================================

print("\n" + "=" * 60)
print("VIDEO TRANSCRIPTION")
print("=" * 60)

transcript_count = 0

for video in video_data.get("videos", []):
    transcript_segments = (
        video.get("insights", {}).get("transcript", [])
    )

    for segment in transcript_segments:
        text = segment.get("text", "")

        if text:
            print(text)
            transcript_count += 1

print("\n" + "=" * 60)
print("Total transcript segments:", transcript_count)


# ============================================================
# STEP 6: Save results locally
# ============================================================

output_dir = Path.cwd() / "video_indexer_results"
output_dir.mkdir(exist_ok=True)

json_path = output_dir / f"{VIDEO_ID}_insights.json"
transcript_path = output_dir / f"{VIDEO_ID}_transcript.txt"

with open(json_path, "w", encoding="utf-8") as file:
    json.dump(video_data, file, indent=2, ensure_ascii=False)

with open(transcript_path, "w", encoding="utf-8") as file:
    for video in video_data.get("videos", []):
        for segment in video.get(
            "insights", {}
        ).get("transcript", []):
            text = segment.get("text", "")
            if text:
                file.write(text + "\n")

print("\nFiles saved successfully:")
print("Insights JSON:", json_path)
print("Transcript TXT:", transcript_path)


Environment file: C:\Users\Laptop Solutions\AI-ML-NETSOL\28-Azure\.env
Video Indexer account ID: 26b25774-e457-4cfc-9cfc-db21dc1fb560
Region: eastus
Video ID: i0xuh58yp3

Azure ARM authentication successful!
ARM token request status: 200
Video Indexer access token generated successfully!

Account verification:
Account ID from .env: 26b25774-e457-4cfc-9cfc-db21dc1fb560
Account ID from token: 26b25774-e457-4cfc-9cfc-db21dc1fb560
Account ID verification passed!

Video indexing HTTP status: 200
Video indexing information retrieved successfully!

VIDEO TRANSCRIPTION
Hello Microsoft, my name is Ipti Saham Akzal and I am currently a second year student of Bachelor of Computer Science at COMSET Saiwal.
So talking about the technical skills that I have learned so far are C, C++, Python and during my recent semester break I have decided to learn front-end development using HTML, CSS and JavaScript.
I started techie courses on platforms like Coursera and YouTube.
These resources enable me to make